# Operational DRY/WET forecast — interactive run

Thin wrapper around `run_forecast.run()` — the same code path as the command line, so results are identical. See the README for the input options (downloaded vs cluster AIFS, NCMRWF portal vs local files).

In [ ]:
import os, sys
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)

import matplotlib.pyplot as plt
import run_forecast
from pipeline import archive, check

check.run_check(os.path.join(REPO, '_scratch'))   # environment / models / feeds

## Choose the forecast

`CONFIG`: `gefs_reduced`, `gefs_full`, `ncmrwf_reduced`, `ncmrwf_full`. `DATE` is the 00Z init date (valid date = DATE + 1 day).

Optional: `AIFS_ZARR` = path to a cluster AIFS-ENS v2 `.zarr` store (otherwise AIFS is downloaded from ECMWF open data); `NCMRWF_FILE` = local NCMRWF plain file (otherwise fetched from the portal, API key required).

In [ ]:
CONFIG = 'gefs_reduced'
DATE = '20260923'
CYCLE = '00'
OUT_DIR = os.path.join(REPO, '_scratch')
AIFS_ZARR = None      # e.g. '/path/to/init_20260923T00.zarr'
NCMRWF_FILE = None    # e.g. '/path/to/20260923.nc' (lag file found next to it)

In [ ]:
fig_out = os.path.join(OUT_DIR, f'{CONFIG}_{DATE}_{CYCLE}.png')
run_forecast.run(CONFIG, DATE, CYCLE, OUT_DIR, fig_out,
                 aifs_zarr=AIFS_ZARR, ncmrwf_file=NCMRWF_FILE)

## Result

The `.npz` record next to the PNG holds the exact probabilities (NaN over sea).

In [ ]:
rec = archive.load(os.path.splitext(fig_out)[0] + '.npz')
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, key in zip(axes, ('p_dry', 'p_wet')):
    im = ax.pcolormesh(rec['lons'], rec['lats'], rec[key], cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f"{key[2:].upper()} probability — {rec['config']} {rec['date']} {rec['cycle']}Z")
    ax.set_aspect('equal')
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()